<center>
<img src="img/logo-hds.png" width="650"/>

***

<center>Lecture 11</center>

***

<center>Reproducible Deployment</center>

<center>Introduction to Time-Series Foundation Models</center>

***

<center>23 November 2026</center>
<center>Rahman Peimankar</center>

## Today's goal: let someone else run your model

Open **Lecture11_mock_training.ipynb** in your practice GitHub repository.


## From your project to a teammate’s result

<img src="img/project-github-docker.svg" width="1200" alt="Students prepare project files and push them to GitHub. A teammate clones the files, builds a Docker image and runs a container to print a confusion matrix.">

GitHub shares the **source files and Dockerfile**. Your teammate creates the image and runs the container on their own computer.

We practise this workflow with the mock classifier first, then apply it to the semester project in Assignment 11.


## Three Docker words

<img src="img/docker-simple.svg" width="1050" alt="Dockerfile recipe, built image, and running container">

**Dockerfile:** the recipe. **Image:** the prepared package. **Container:** the package running.

Docker helps package Python, the packages and our project files. It does not improve the classifier's accuracy.

## Train once, then save

**Notebook section 1 · first Python cell:** six invented training rows and a one-split decision tree.

```python
model = DecisionTreeClassifier(max_depth=1, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, folder / "model.joblib")
test.to_csv(folder / "test.csv", index=False)
```

Inputs: mock ECG spread and age. Labels: invented classes 0 and 1.

`test` contains six different mock rows and their known labels. The saved model will evaluate these rows without training again.

## Compare predictions with known answers

**Notebook section 2 · second Python cell:** creates and runs `run_project.py` locally.

```python
model = joblib.load("model.joblib")
test = pd.read_csv("test.csv")

X_test = test[["ecg_spread", "age"]]
y_test = test["label"]
predictions = model.predict(X_test)

matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
```

The script prints a labelled table. **Rows = actual; columns = predicted.**


## Three Python cells create six files

**Notebook section 3 · third Python cell:** finishes the Docker files.

| File | Purpose |
|---|---|
| model.joblib | The fitted decision tree |
| test.csv | Test inputs and their known labels |
| run_project.py | Predict and print the confusion matrix |
| requirements.txt | List Python packages and versions |
| Dockerfile | Describe how to package the program |
| README.md | Commands and expected confusion matrix |

All files are inside `docker_project`. No separate demo folder is needed.

## requirements.txt is a shopping list for Python

An example line is:

```text
scikit-learn==1.8.0
```

It says which package to install and which version to use. Your notebook writes the versions from your own environment.

Docker installs the list with:

```text
python -m pip install -r requirements.txt
```

The list contains software packages, not the trained model or data.

## Build the package

**Notebook step 4 · Terminal:** start at the repository root.

```text
cd docker_project
docker build -t semester-project .
```

`cd` enters the folder. `build` creates an image from the Dockerfile.

`-t semester-project` gives it a name. The final dot means “use this folder.”

Docker Desktop must be running. The first build downloads Python and packages, so wait until it finishes.

## Run and check the confusion matrix

Stay inside `docker_project` and run:

```text
docker run --rm semester-project
```

Expected output (the same as the notebook):

```text
          Predicted 0  Predicted 1
Actual 0            2            1
Actual 1            1            2
```

`run` starts the container; `--rm` removes it after the script exits. The image remains. Rebuild after changing files.

## Share the files through GitHub

**Notebook step 5 · Terminal:**

```text
cd ..
git add Lecture11_mock_training.ipynb docker_project
git status
git commit -m "Add Docker confusion matrix example"
git push
```

Go up one folder, select the project files, review them, save a local revision, then send it to GitHub.

Use your existing configured repository and sign-in. You share the files and recipe; GitHub does not start the program.

## Someone else downloads and runs it

**Notebook section 5, continued:** your teammate opens a terminal in a new folder.

```text
git clone https://github.com/OWNER/REPOSITORY.git
cd REPOSITORY/docker_project
docker build -t semester-project .
docker run --rm semester-project
```

Replace OWNER and REPOSITORY. `clone` downloads the project. The next commands enter the folder, build the image and run it.

Your teammate needs Git, Docker and repository access. They compare the confusion matrix with the README.

## A brief introduction to foundation models

- A foundation model learns useful patterns from a large collection of data before we use it for a particular task. This is called **pretraining**.

- For ECG, we can reuse a pretrained model to produce features describing a signal.

- A classifier, including logistic regression, can then learn from these features and our labels.

- Foundation models are also machine learning models. They do not automatically replace simpler classifiers.

## Why might we use one?

**Possible benefit:** learned features may capture ECG shapes that simple summaries miss.

**Another reason:** reuse previous learning when our labelled dataset is limited.

**Trade-off:** larger models often need more memory and computation, and may be harder to explain.

If a simple model performs well, it may be the better choice. 

## ECG classification: the basic idea

<img src="img/ecg-simple.svg" width="1050" alt="ECG goes through a pretrained feature extractor, then logistic regression predicts a class">

- The pretrained model supplies learned features. Logistic regression learns the class labels from our examples.

- MOMENT is one time-series model to explore: [authors’ introduction](https://moment-timeseries-foundation-model.github.io/) · [paper](https://arxiv.org/abs/2402.03885).


<img src="img/fm-hds-new.png" width="650"/>

## Exercise 1 · Explain it to a partner

Use your own words:

1. What is a Docker image?
2. What happens when we run a container?
3. What does requirements.txt tell Docker?

Then explain the difference between **git push** and **docker run**.

## Exercise 2 · Check the handoff

Open the README generated by the mock notebook.

1. Find the build and run commands and expected matrix.
2. Identify a correct prediction count and an error count.
3. Why must `label` stay out of the input features?
4. Does matching this matrix prove accuracy on new patients?

Assignment 11 applies the workflow to your own project and records the actual local, Docker and fresh-clone results.

## What students should take away

Docker packages our evaluation script and its software environment.

GitHub shares the files. A teammate clones them, builds an image and runs the evaluation.

**Final check:** Could a teammate follow your README and reproduce your result?

[Docker build](https://docs.docker.com/reference/cli/docker/buildx/build/) · [Confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)